# Quantum-limited imaging using diffractive optical neural networks — Fig. 3

Companion notebook for

> A. Warke, A. Zhang, A. I. Lvovsky, *Quantum-limited imaging using diffractive optical neural networks*, [arXiv:2608.12300](https://arxiv.org/abs/2608.12300).

**Fig. 3 — attainability of the quantum limit for many parameters.** All $M$ cosine amplitudes $a_m$, $m = 1 \ldots M$, are estimated jointly. The field of view grows with $M$: $L(M)$ is chosen so that $M$ is the *natural* number of modes the aperture admits there ($m_{\rm cut} = \lfloor 4\,\mathrm{NA}\,L/\lambda \rfloor = M$), and the optics are rebuilt per $M$.

- **panel (a)** — per-photon total variance $\mathrm{Tr}\,F^{-1}$ vs $M$ (field of view on the top axis): QCRB, NHCRB from the Nagaoka–Hayashi SDP, direct-imaging CRB, and the DONN CRB with its Monte-Carlo variance — the DONN saturates the NHCRB (DONN/NH $= 1.000$);
- **panel (b)** — the $M = 15$ point opened up into per-parameter variances $[F^{-1}]_{jj}$: QCRB / DONN / DI bars for each mode, with Monte-Carlo markers and the coherent cutoff marked. Consistency checks: the bars sum to the panel-(a) point, and MC matches Fisher within the expected scatter.

**Precision split:** the DONN mask search runs in `float32`/`complex64` (large GPU speed-up); every reported bound — QCRB, DI, NHCRB — and all Monte Carlo are evaluated in NumPy float64.

**Outputs:** `figures/WZL_Fig3.png` / `.svg`, `data/WZL_Fig3a.npz` (panel-a sweep), and `data/WZL_Fig3b_M15.npz` (panel-b cache, including the trained masks).

**Requirements:** `numpy`, `torch`, `matplotlib`, `cvxpy` (Clarabel). A CUDA GPU is used automatically if available. Cells 6 and 7 are marked **[slow]** (one DONN training + one SDP per $M$; panel (b) retrains at $M = 15$ replaying the sweep seed, or reloads its saved masks if the cache exists). The final cell, marked **[fast]**, only reads the two saved `.npz` files — re-run it freely for cosmetic changes.

In [ ]:
# 1. CONFIG
import os, time, json
import numpy as np, cvxpy as cp, torch, torch.nn as nn, torch.optim as optim

# device / precision
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RDTYPE = torch.float32        # real dtype    -> float32 for a large GPU speed-up
CDTYPE = torch.complex64      # complex dtype -> must match RDTYPE

torch.set_default_dtype(RDTYPE)
if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = False   # keep full float32 precision
    print(f"training on {torch.cuda.get_device_name(0)}  ({RDTYPE})", flush=True)
else:
    print("CUDA not available - training on CPU", flush=True)

torch.manual_seed(0); np.random.seed(0)

def _sync(device=DEVICE):
    if device.type == "cuda": torch.cuda.synchronize()

# tunables
RESTARTS    = 1
EP0, EPS    = 1500, 1500      # epochs: initial 2-plane stage / each appended pair
LR0         = 0.05
POLISH      = True            # L-BFGS polish of the best Adam solution
PLANES_MULT = 2               # n_planes = even(PLANES_MULT * M), min 4
EIG_THRESH  = 1e-9            # eigenbasis truncation -> K.  QCRB converged from ~1e-5;
                              # 1e-3 gives K=33 (block 528) vs 39 (624) at L=5.5.
NH_SOLVER   = "CLARABEL"           # CLARABEL is exact but memory-bound at this K
NH_MAX_M    = 15              # skip the (slow) SDP above this M
EPS_CUT     = 0.5             # m_cut = M + EPS_CUT; alternative to floor
M_LIST      = list(range(2, 16))
M_PANEL     = 15              # which point of (a) panel (b) opens up
MC_RUNS     = 5000
MC_PHOTONS  = 100000

SWEEP_NPZ = "data/WZL_Fig3a.npz"
PANEL_NPZ = f"data/WZL_Fig3b.npz"
FIGNAME   = "figures/WZL_Fig3"

C_QCRB, C_DI, C_DONN = "crimson", "0.4", "darkorange"
FONTSZ  = 16
SHOW_MC = True

In [ ]:
# 2. OPTICS
def optics_exact_1d(NA=1.4, lam=0.540, L=5.5, N=256, det_span=2, tol=1e-12,
                    normalize_det=False):
    dx = L/N
    nc = NA/lam
    x_mid = (np.arange(N)+0.5)*dx

    # detector / propagation grid, centred on the object, det_span*L wide
    Nd = det_span*N
    x_det = (np.arange(Nd)+0.5)*dx - (Nd-N)//2*dx
    psi  = np.sqrt(2*nc)*np.sinc(2*nc*(x_det[None,:]-x_mid[:,None]))*np.sqrt(dx)
    P_wf = psi**2
    capture = P_wf.sum(1)                     # light caught per emitter
    if normalize_det:
        psi = psi/np.sqrt(capture)[:,None]; P_wf = psi**2

    G = psi @ psi.T
    ev, V = np.linalg.eigh(G); keep = ev > tol*ev.max()
    T = V[:,keep]*np.sqrt(ev[keep])           # (N,K): rho = T.T@(w[:,None]*T)

    return dict(NA=NA, lam=lam, L=L, N=N, Nd=Nd, dx=dx, nu_coh=nc, nu_incoh=2*nc,
                x_mid=x_mid, x_det=x_det, T=T, K=int(keep.sum()),
                capture=capture, psi=psi, P_wf=P_wf)

def cos_derivs(m_list, opt, a0=1.0):
    L,N,dx,x = opt['L'],opt['N'],opt['dx'],opt['x_mid']
    ft = a0*N*dx; w = np.full(N,a0)/ft*dx
    dw = np.zeros((len(m_list),N))
    for i,m in enumerate(m_list):
        C = np.cos(np.pi*m/L*x); dft = np.sum(C)*dx
        dw[i] = (C*ft-a0*dft)/ft**2*dx
    return w, dw

def eigenbasis(w, dwl, opt, thresh=EIG_THRESH):
    T = opt['T']                                 # exact optics returns T, not `outers`
    rho = T.T @ (w[:,None]*T)
    drf = [T.T @ (dw[:,None]*T) for dw in dwl]
    ev, U = np.linalg.eigh(rho); keep = ev > thresh*ev.max()
    lam, Uk = ev[keep], U[:,keep]
    return np.diag(lam), [Uk.T@dr@Uk for dr in drf], len(lam)

def L_for_M(M, NA=1.4, lam=0.540, eps=EPS_CUT):
    return (M + eps) * lam / (4*NA)

def modes_for_M(M, opt=None):
    return list(range(1, M+1))

def nu_of(modes, opt):
    return np.asarray(modes)/(2*opt['L'])/opt['nu_coh']

In [ ]:
# 3. BOUNDS
def fim_di(w, dwl, opt, Nph=1.0):
    P = opt['P_wf']; M = len(dwl)
    u = Nph*(w[:,None]*P).sum(0); du = [Nph*(dw[:,None]*P).sum(0) for dw in dwl]
    return np.array([[np.sum(du[a]*du[b]/(u+1e-15)) for b in range(M)] for a in range(M)])

def qfi_matrix(re, drl, Nph=1.0):
    lam = np.diag(re); M = len(drl); den = lam[:,None]+lam[None,:]
    ok = den > 1e-12*lam.max(); inv = np.zeros_like(den); inv[ok] = 1/den[ok]
    return np.array([[Nph*2*np.sum(drl[a]*drl[b]*inv).real for b in range(M)] for a in range(M)])

def per_param_var(F):
    return np.diag(np.linalg.inv(F))

crb_di  = lambda w, dwl, opt, Nph=1.0: np.trace(np.linalg.inv(fim_di(w,dwl,opt,Nph)))
crb_qfi = lambda re, drl, Nph=1.0:     np.trace(np.linalg.inv(qfi_matrix(re,drl,Nph)))

def fim_from_detector(w, dw, P):
    u0 = w@P; D = dw@P
    return (D/np.sqrt(u0)) @ (D/np.sqrt(u0)).T

def crb_nhcrb(re, drl, K, Nph=1.0, solver=NH_SOLVER, max_iters=60000, eps=1e-9):
    M = len(drl)
    if K < 2: return np.inf, "trivial"
    dn  = np.array([np.linalg.norm(dd) for dd in drl])
    drt = [dd/nn for dd,nn in zip(drl,dn)]
    wgt = 1.0/dn**2
    Xs = [cp.Variable((K,K),symmetric=True) for _ in range(M)]
    Ls = {(j,k):cp.Variable((K,K),symmetric=True) for j in range(M) for k in range(j,M)}
    gL = lambda j,k: Ls[(j,k)] if j<=k else Ls[(k,j)].T
    cons = [cp.trace(drt[j]@Xs[k])==(1 if j==k else 0) for j in range(M) for k in range(M)]
    cons.append(cp.bmat([[gL(j,k) for k in range(M)]+[Xs[j]] for j in range(M)]
                        + [[Xs[m].T for m in range(M)]+[np.eye(K)]]) >> 0)
    prob = cp.Problem(cp.Minimize(sum(wgt[j]*cp.trace(re@Ls[(j,j)]) for j in range(M))), cons)
    for sol in ([cp.CLARABEL,cp.SCS] if solver=="CLARABEL" else [cp.SCS]):
        try:
            kw = {} if sol==cp.CLARABEL else dict(max_iters=max_iters,eps=eps)
            prob.solve(solver=sol,verbose=False,**kw)
            if prob.value is not None and np.isfinite(prob.value):
                return prob.value/Nph, prob.status
        except Exception: continue
    return np.inf, "failed"

In [ ]:
# 4. DONN (GPU preferred)
class MPLC(nn.Module):
    def __init__(self, opt, m_list, n_planes, a0=1.0, init=None, device=DEVICE):
        super().__init__()
        Nd = opt['psi'].shape[1]                  # propagation grid, NOT opt['N']
        self.n = n_planes; self.M = len(m_list)
        self.register_buffer('psi', torch.tensor(opt['psi'], dtype=CDTYPE, device=device))
        w, dw = cos_derivs(m_list, opt, a0)
        self.register_buffer('w',  torch.tensor(w,  dtype=RDTYPE, device=device))
        self.register_buffer('dw', torch.tensor(dw, dtype=RDTYPE, device=device))
        if init is None:
            # drawn on the CPU in float64 so the seed sequence is identical on CPU and
            # GPU and independent of RDTYPE, then moved to `device`
            init = [torch.randn(Nd, dtype=torch.float64)*0.5 for _ in range(n_planes)]
        self.masks = nn.ParameterList([
            nn.Parameter(p.detach().clone().to(device=device, dtype=RDTYPE)) for p in init])

    def _U(self, psi):
        for k in range(self.n):
            psi = psi*torch.exp(1j*self.masks[k].to(CDTYPE)).unsqueeze(0)
            psi = torch.fft.fftshift(
                    torch.fft.fft(torch.fft.ifftshift(psi, dim=1), norm='ortho'), dim=1)
        return psi

    def forward(self):
        Pd = torch.abs(self._U(self.psi))**2
        u  = torch.clamp((self.w.unsqueeze(1)*Pd).sum(0), min=1e-12)
        du = torch.einsum('mj,jd->md', self.dw, Pd)
        eye = torch.eye(self.M, dtype=Pd.dtype, device=Pd.device)   # device-aware
        F = (du/u) @ du.T + 1e-12*eye
        return torch.trace(torch.linalg.inv(F))

def _adam(model, epochs, lr):
    o = optim.Adam(model.parameters(), lr=lr)
    s = optim.lr_scheduler.CosineAnnealingLR(o, T_max=epochs)
    best = np.inf; bm = None
    for _ in range(epochs):
        o.zero_grad(); v = model(); v.backward(); o.step(); s.step()
        vi = v.item()
        if vi < best: best = vi; bm = [p.detach().clone() for p in model.masks]
    return best, bm

def _lbfgs(model, rounds=50):
    o = optim.LBFGS(model.parameters(), lr=0.3, max_iter=20, line_search_fn='strong_wolfe')
    best = np.inf; bm = None
    for _ in range(rounds):
        def cl(): o.zero_grad(); v = model(); v.backward(); return v
        try: v = o.step(cl)
        except Exception: break
        v = float(v.detach())
        if v < best: best = v; bm = [p.detach().clone() for p in model.masks]
    return best, bm

def n_planes_for_M(M, mult=PLANES_MULT):
    n = max(mult*M, 4); return n if n%2==0 else n+1

def fresh_donn(opt, m_list, n_planes, restarts=RESTARTS, ep0=EP0, eps=EPS,
               lr0=LR0, polish=POLISH, seed0=0, verbose=False, device=DEVICE):
    Nd = opt['psi'].shape[1]                      # propagation grid, NOT opt['N']
    bestO = np.inf; bestM = None
    for r in range(restarts):
        torch.manual_seed(seed0 + 1000*r)
        _sync(device); t0 = time.time()
        model = MPLC(opt, m_list, 2, device=device)
        b, mk = _adam(model, ep0, lr0); n = 2
        if device.type == "cuda": del model; torch.cuda.empty_cache()
        if verbose: print(f"  restart {r}: n=2  Tr[F^-1]={b:.3f}  ({time.time()-t0:.0f}s)", flush=True)
        while n < n_planes:
            init = list(mk) + [torch.randn(Nd, dtype=torch.float64)*1e-3,
                               torch.randn(Nd, dtype=torch.float64)*1e-3]
            n += 2
            model = MPLC(opt, m_list, n, init=init, device=device)
            b, mk = _adam(model, eps, lr0*0.6)
            if device.type == "cuda": del model; torch.cuda.empty_cache()
            if verbose: print(f"  restart {r}: n={n:<3d} Tr[F^-1]={b:.3f}  ({time.time()-t0:.0f}s)", flush=True)
        if polish:
            model = MPLC(opt, m_list, n_planes, init=mk, device=device)
            bp, mk2 = _lbfgs(model)
            if device.type == "cuda": del model; torch.cuda.empty_cache()
            if bp < b and mk2 is not None:
                b, mk = bp, mk2
                if verbose: print(f"  restart {r}: LBFGS polish -> {b:.3f}", flush=True)
        _sync(device)
        if b < bestO: bestO = b; bestM = mk
    return bestO, bestM

def donn_detector(opt, masks):
    psi = torch.tensor(opt['psi'], dtype=torch.complex128)
    for mk in masks:
        m = mk.detach().to(device='cpu', dtype=torch.float64)
        psi = psi * torch.exp(1j*m.to(torch.complex128)).unsqueeze(0)
        psi = torch.fft.fftshift(
                torch.fft.fft(torch.fft.ifftshift(psi, dim=1), norm='ortho'), dim=1)
    return (torch.abs(psi)**2).numpy()

In [ ]:
# 5. MONTE CARLO
def mc_pervar_tr(w, dw, P, N=MC_PHOTONS, T=MC_RUNS, seed=0):
    u0 = w@P; D = dw@P; Finv = np.linalg.inv(fim_from_detector(w, dw, P))
    rng = np.random.default_rng(seed)
    n = rng.poisson(N*u0, size=(T,len(u0))); est = ((n/u0)@D.T)@Finv.T/N
    cov = np.cov(est.T) if D.shape[0] > 1 else np.array([[est[:,0].var(ddof=1)]])
    return np.trace(cov)*N

def mc_per_param(w, dw, P, N=MC_PHOTONS, T=MC_RUNS, seed=0):
    """Per-parameter variances, times N -> comparable to per_param_var(F)."""
    u0 = w@P; D = dw@P; Finv = np.linalg.inv(fim_from_detector(w, dw, P))
    rng = np.random.default_rng(seed)
    n = rng.poisson(N*u0, size=(T,len(u0))); est = ((n/u0)@D.T)@Finv.T/N
    return est.var(axis=0, ddof=1)*N

In [ ]:
# 6. RUN panel (a)   [slow]
def sweep_attain(M_list=M_LIST, verbose=True):
    out = {k:[] for k in ['M','L','nplanes','K','di','qcrb','nhcrb','donn','mc_di','mc_donn','modes']}
    if verbose:
        print(f"{'M':>3}{'L(um)':>8}{'npl':>4}{'K':>4}{'DI':>9}{'QCRB':>8}{'NHCRB':>9}{'DONN':>8}"
              f"{'DONN_MC':>9}{'DI/DONN':>8}{'D/NH':>7}{'t':>7}  NHstatus", flush=True)
    for M in M_list:
        t0 = time.time()
        L = L_for_M(M); opt = optics_exact_1d(L=L)      # FOV grows with M
        modes = modes_for_M(M); npl = n_planes_for_M(M)
        w,dw = cos_derivs(modes, opt); dwl = list(dw)
        re,dr,K = eigenbasis(w, dwl, opt)
        cdi, cq = crb_di(w, dwl, opt), crb_qfi(re, dr)
        b, mk = fresh_donn(opt, modes, npl, seed0=M)      # per-M, no carry
        Pd = donn_detector(opt, mk)
        v_di = mc_pervar_tr(w, dw, opt['P_wf']); v_dn = mc_pervar_tr(w, dw, Pd)
        cn, st = crb_nhcrb(re, dr, K) if M <= NH_MAX_M else (np.nan, "skipped")
        for k,v in zip(out, [M,L,npl,K,cdi,cq,cn,b,v_di,v_dn,modes]): out[k].append(v)
        if verbose:
            dnh = b/cn if np.isfinite(cn) else np.nan
            print(f"{M:3d}{L:8.3f}{npl:4d}{K:4d}{cdi:9.2f}{cq:8.2f}{cn:9.2f}{b:8.2f}"
                  f"{v_dn:9.2f}{cdi/b:8.2f}{dnh:7.3f}{time.time()-t0:7.0f}  {st}", flush=True)
    for k in out:
        if k != 'modes': out[k] = np.array(out[k], dtype=float)
    return out

print(f"FOV grows with M:  L(M) = ⌊ M * lam/(4 NA) ⌋;  "
      f"{1000*0.540/(4*1.4):.1f} nm per mode")
for M in [M_LIST[0], M_LIST[-1]]:
    L = L_for_M(M); o = optics_exact_1d(L=L)
    nu = nu_of(modes_for_M(M), o)
    print(f"  M={M:2d}: L={L:.3f} um  K_T={o['K']}  capture={o['capture'].mean():.4f}  "
          f"nu/nuc=[{nu.min():.3f},{nu.max():.3f}]", flush=True)

t0 = time.time()
res = sweep_attain()
np.savez(SWEEP_NPZ, **{k:v for k,v in res.items() if k!='modes'})
print(f"\nsaved {SWEEP_NPZ}   [{time.time()-t0:.0f}s]", flush=True)

In [ ]:
# 7. RUN panel (b)   [slow]
L_panel = L_for_M(M_PANEL); opt = optics_exact_1d(L=L_panel)
modes = modes_for_M(M_PANEL); nu = nu_of(modes, opt)
w, dw = cos_derivs(modes, opt); dwl = list(dw)
re, dr, K = eigenbasis(w, dwl, opt); npl = n_planes_for_M(M_PANEL)
print(f"panel (b): M={M_PANEL}  L={L_panel:.3f} um  K={K}  n_planes={npl} ({npl//2} blocks)")
print("modes:", modes, "\nnu:", np.round(nu,3), flush=True)

var_qcrb = per_param_var(qfi_matrix(re, dr))
var_di   = per_param_var(fim_di(w, dwl, opt))

if os.path.exists(PANEL_NPZ):
    z = np.load(PANEL_NPZ, allow_pickle=True)
    masks = [torch.tensor(m) for m in z["masks"]]; donn_trace = float(z["donn_trace"])
    print(f"loaded masks from {PANEL_NPZ}  (Tr[F^-1]={donn_trace:.3f})", flush=True)
else:
    t0 = time.time()
    donn_trace, masks = fresh_donn(opt, modes, npl, seed0=M_PANEL, verbose=True)
    print(f"trained in {time.time()-t0:.0f}s   Tr[F^-1]={donn_trace:.3f}", flush=True)

Pd = donn_detector(opt, masks)
var_donn = per_param_var(fim_from_detector(w, dw, Pd))
mc_donn  = mc_per_param(w, dw, Pd,          seed=M_PANEL)   if SHOW_MC else np.full(M_PANEL,np.nan)
mc_di    = mc_per_param(w, dw, opt['P_wf'], seed=M_PANEL+1) if SHOW_MC else np.full(M_PANEL,np.nan)

np.savez(PANEL_NPZ, M=M_PANEL, L=L_panel, modes=np.asarray(modes), nu=nu, K=K, n_planes=npl,
         var_qcrb=var_qcrb, var_di=var_di, var_donn=var_donn,
         mc_donn=mc_donn, mc_di=mc_di, donn_trace=donn_trace,
         masks=np.stack([m.detach().cpu().numpy() for m in masks]))

hdr = f"{'j':>3}{'m':>5}{'nu':>8}{'QCRB':>10}{'DONN':>10}{'DI':>10}{'DONN/Q':>9}{'DI/Q':>8}"
print("\n"+hdr+"\n"+"-"*len(hdr))
for j,(m,n_,q,dn,di) in enumerate(zip(modes,nu,var_qcrb,var_donn,var_di),1):
    print(f"{j:3d}{m:5d}{n_:8.3f}{q:10.2f}{dn:10.2f}{di:10.2f}{dn/q:9.2f}{di/q:8.2f}")
print("-"*len(hdr))
print(f"{'sum':>16}   {var_qcrb.sum():9.2f}{var_donn.sum():10.2f}{var_di.sum():10.2f}")

# --- checks: bars must sum to the panel-(a) point; MC must match Fisher ---
s = np.load(SWEEP_NPZ); i = int(np.where(s["M"]==M_PANEL)[0][0])
for lbl, mine, key, lim in [("QCRB",var_qcrb.sum(),"qcrb",1e-6),
                            ("DI",  var_di.sum(),  "di",  1e-6),
                            ("DONN",var_donn.sum(),"donn",1e-2)]:
    ref = float(s[key][i]); rel = abs(mine-ref)/ref
    print(f"  {'ok ' if rel<lim else 'CHECK'} {lbl:5s} here={mine:9.3f} sweep={ref:9.3f} rel={rel:.2e}")
if SHOW_MC:
    tol = np.sqrt(2/MC_RUNS)
    for lbl, fish, mc in [("DONN",var_donn,mc_donn), ("DI",var_di,mc_di)]:
        rel = np.abs(mc-fish)/fish
        print(f"  {'ok ' if rel.max()<3*tol else 'CHECK'} {lbl:5s} max MC deviation {100*rel.max():5.2f}% "
              f"(expected scatter {100*tol:.1f}%)")

In [ ]:
# 8. PLOT  Fig. 3   [fast]
# Reads only the two .npz files
import matplotlib.pyplot as plt
from matplotlib.ticker import LogLocator, NullFormatter, MultipleLocator
from matplotlib.transforms import blended_transform_factory
from pathlib import Path

def panel_a(ax, npz_file=SWEEP_NPZ):
    d = np.load(npz_file); M = np.asarray(d["M"])
    ax.plot(M, d["qcrb"], "-", color=C_QCRB, lw=2, label="QCRB")
    ok = np.isfinite(d["nhcrb"])
    ax.plot(M[ok], d["nhcrb"][ok], "s", mfc="none", mec="C0", mew=1.8, ms=11, label="NHCRB")
    ax.plot(M, d["di"],      "o-", color=C_DI,   lw=2, ms=6, label="Direct imaging")
    ax.plot(M, d["donn"],    "D-", color=C_DONN, lw=2, ms=7, label="DONN CRB")
    ax.plot(M, d["mc_donn"], "+",  color="k",    ms=11, mew=1.6, label="DONN (MC variance)")
    ax.tick_params(labelsize=FONTSZ); ax.set_yscale("linear")
    ax.set_xlabel(r"Number of jointly-estimated amplitudes $(M)$", fontsize=FONTSZ)
    ax.set_ylabel(r"Per-photon variance", fontsize=FONTSZ)
    ax.yaxis.set_major_locator(MultipleLocator(100)); ax.set_xticks(M)
    ax.grid(True, which="both", ls="-.", alpha=0.3)
    ax.legend(fontsize=13, loc="upper left")
    if "L" in d.files:                      # secondary axis: the FOV each M is measured in...
        axL = ax.secondary_xaxis("top")
        axL.set_xticks(M); axL.set_xticklabels([f"{v:.2f}" for v in d["L"]],
                                               fontsize=FONTSZ-2, rotation=45)
        axL.set_xlabel(r"Field of view $L$ ($\mu$m)", fontsize=FONTSZ-2, labelpad=6)

def panel_b(ax, cache=PANEL_NPZ):
    z = np.load(cache, allow_pickle=True)
    nu_, vq, vdn, vdi = z["nu"], z["var_qcrb"], z["var_donn"], z["var_di"]
    mcd, mci = z["mc_donn"], z["mc_di"]
    M = len(nu_); x = np.arange(M); wbar = 0.27
    ax.bar(x-wbar, vq,  wbar, color=C_QCRB, label="QCRB",            zorder=3)
    ax.bar(x,      vdn, wbar, color=C_DONN, label="DONN CRB", zorder=3)
    ax.bar(x+wbar, vdi, wbar, color=C_DI,   label="Direct imaging",  zorder=3)
    if SHOW_MC and np.isfinite(mcd).all():
        ax.plot(x,      mcd, "+", color="k", ms=9, mew=1.4, zorder=5, label="DONN (MC variance)")
        ax.plot(x+wbar, mci, "x", color="k", ms=6, mew=1.2, zorder=5, label="Direct imaging (MC variance)")
    ax.set_yscale("log")
    ax.set_ylim(0.6*min(vq.min(),vdn.min(),vdi.min()),
                8.0*max(vq.max(),vdn.max(),vdi.max()))
    ax.set_xlim(-0.6, M-0.4)
    if nu_.min() < 1.0 < nu_.max():                       # coherent cutoff at nu = 1
        i  = int(np.searchsorted(nu_, 1.0))
        xc = i-1 + (1.0-nu_[i-1])/(nu_[i]-nu_[i-1])
        ax.axvline(xc, ls=":", lw=1.6, color="0.35", zorder=2)
        ax.text(xc+0.12, 0.58, "coherent\ncutoff", fontsize=FONTSZ-4, color="0.35",
                ha="left", va="center", linespacing=1.1,
                transform=blended_transform_factory(ax.transData, ax.transAxes))
    ax.set_xticks(x)
    ax.set_xticklabels([f"{v:.2f}" for v in nu_], fontsize=FONTSZ-4, rotation=45, ha="right")
    ax.tick_params(axis="y", labelsize=FONTSZ)
    ax.set_xlabel(r"Normalized spatial frequency", fontsize=FONTSZ)
    ax.set_ylabel(r"Per-photon $[F^{-1}]_{jj}$", fontsize=FONTSZ)
    ax.yaxis.set_minor_locator(LogLocator(base=10.0, subs=np.arange(2,10)*0.1, numticks=20))
    ax.yaxis.set_minor_formatter(NullFormatter())
    ax.grid(True, axis="y", which="both", ls="-.", alpha=0.25, zorder=0)
    ax.legend(fontsize=FONTSZ-3, loc="upper left", ncol=3, framealpha=0.95)

fig, (ax_a, ax_b) = plt.subplots(2, 1, figsize=(10.5, 10), dpi=500,
                                 gridspec_kw={"height_ratios":[1.05, 0.95], "hspace":0.35})
panel_a(ax_a)
panel_b(ax_b)

out = Path(FIGNAME)
fig.savefig(out.with_suffix(".png"), dpi=500, bbox_inches="tight")
fig.savefig(out.with_suffix(".svg"),            bbox_inches="tight")
print(f"saved {out.with_suffix('.png')} and {out.with_suffix('.svg')}")
plt.show()